# PM Case Coach — evaluation
AI-assisted starter code (Claude, Anthropic). Review and edit before committing.

What this notebook does:
1. Loads `test_set.csv` (written **before** any run).
2. Runs each planted-weakness answer through feedback prompt **V1** (model's own knowledge) and **V2** (retrieved rubric + few-shot), 5 times each.
3. Checks **with code** whether the planted category appears in the report. That is our metric and our deterministic check (§5.5 items 1 and 3).
4. Records tokens per call for the cost model (§5.6).
5. Measures false alarms on strong answers: the failure our metric cannot see (§5.5 item 5).

Run it from the `eval/` folder so the paths below work.

In [ ]:
# Setup: make the app's files importable and load the key from a secret, never pasted here.
import os, sys, time
import pandas as pd
sys.path.append("..")
os.chdir("..")  # the app's code reads data/ relative to the repo root

from google import genai
from google.genai import types
from pydantic import ValidationError
from config import CHAT_MODEL
from prompts import FEEDBACK_PROMPTS
from schemas import FeedbackReport
from retrieval import search

# Colab: Secrets panel (key icon) -> add GEMINI_API_KEY, then use:
#   from google.colab import userdata; api_key = userdata.get("GEMINI_API_KEY")
# VS Code: set the environment variable GEMINI_API_KEY before starting Jupyter.
api_key = os.environ.get("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)

test = pd.read_csv("eval/test_set.csv")
weak = test[test.input_type == "planted_weakness"]
strong = test[test.input_type == "strong_answer"]
print(len(weak), "planted-weakness rows |", len(strong), "strong rows")

In [ ]:
def run_feedback(version, transcript):
    """Same prompt, model and schema as app.py. Returns (report or None, tokens)."""
    prompt = FEEDBACK_PROMPTS[version]
    if "{rubric}" in prompt:
        hits = search(client, transcript[-3000:], k=4)
        prompt = prompt.replace("{rubric}", "\n\n".join(h["text"] for h in hits))
    resp = client.models.generate_content(
        model=CHAT_MODEL, contents=transcript,
        config=types.GenerateContentConfig(
            system_instruction=prompt, temperature=0.2,
            response_mime_type="application/json", response_schema=FeedbackReport))
    try:
        report = FeedbackReport.model_validate_json(resp.text)
    except ValidationError:
        report = None
    return report, resp.usage_metadata.total_token_count

# Try it on ONE row first, before looping (directives §5.4: develop against two rows).
r, t = run_feedback("V2", weak.iloc[0].transcript)
print(t, "tokens"); print(r)

## The comparison: V1 vs V2, 5 runs each
`found` = did the report name the planted category (our metric). `parsed` = did the JSON parse (deterministic check).
To limit calls, set `ROWS` to a subset (e.g. 8 rows) and say in the report which and why.

In [ ]:
ROWS = weak            # or weak.head(8)
RUNS = 5              # directives §5.5: at least 5 per version, ideally 10

rows = []
for _, t in ROWS.iterrows():
    for version in ["V1", "V2"]:
        for run in range(RUNS):
            report, tokens = run_feedback(version, t.transcript)
            found = report is not None and any(w.category == t.planted_category for w in report.weaknesses)
            rows.append(dict(id=t.id, version=version, run=run,
                             parsed=report is not None, found=found, tokens=tokens))
            time.sleep(5)  # stay under the free-tier rate limit

results = pd.DataFrame(rows)
results.to_csv("eval/results.csv", index=False)
results.groupby("version")[["found", "parsed", "tokens"]].agg(["mean", "std"])

## The failure the metric cannot see: invented weaknesses
On answers written to have **no** weakness, every weakness the app lists is a false alarm.

In [ ]:
fa = []
for _, t in strong.iterrows():
    report, tokens = run_feedback("V2", t.transcript)
    fa.append(dict(id=t.id, false_alarms=len(report.weaknesses) if report else None, tokens=tokens))
    time.sleep(5)
pd.DataFrame(fa)

## Write your interpretation here (in your own words, §9)
- Metric for V1 and V2, with spread:
- Tokens per call for each version, and whether V2's gain is worth it:
- Did V2 reason better, or recall from training data? What is the evidence?
- False alarms on strong answers, and what that means: